### DATA PREPROCESSING


1. LOAD + CLEAN COLUMNS

In [14]:
import pandas as pd

phishing = pd.read_csv('phishing_dataset.csv')
legit = pd.read_csv('url_dataset.csv')

# --- CLEAN LEGIT DATASET ---
# Drop old 'type' column if it exists
if 'type' in legit.columns:
    legit = legit.drop(columns=['type'])

# Ensure only correct label column exists
legit['Type'] = 0
phishing['Type'] = 1

# --- VERIFY ---
print("Phishing columns:", phishing.columns)
print("Legit columns:", legit.columns)

Phishing columns: Index(['url', 'Type'], dtype='object')
Legit columns: Index(['url', 'Type'], dtype='object')


2. CHECK NULLS.

In [15]:
print(phishing.isnull().sum())
print(legit.isnull().sum())

url     0
Type    0
dtype: int64
url     0
Type    0
dtype: int64


3. VERIFY SCHEMA

In [16]:
print(phishing.columns)
print(legit.columns)

Index(['url', 'Type'], dtype='object')
Index(['url', 'Type'], dtype='object')


4. MERGE DATASETS

In [17]:
# --- MERGE DATASETS ---
urldata = pd.concat([phishing, legit], ignore_index=True)

print("Merged dataset size:", len(urldata))

Merged dataset size: 504983


5. NORAMLIZE + REMOVE DUPLICATES + SHUFFLE

In [18]:
# --- NORMALIZE URLs ---
def normalize_url(url):
    url = url.lower().strip()
    url = url.replace('\n', '').replace('\r', '')
    url = url.replace("www.", "")
    if url.endswith('/'):
        url = url[:-1]
    return url

urldata['url'] = urldata['url'].apply(normalize_url)

# --- REMOVE DUPLICATES ---
before = len(urldata)

urldata = urldata.drop_duplicates(subset='url')

after = len(urldata)

print(f"Removed {before - after} duplicate URLs")

# --- CHECK CLASS BALANCE ---
print("\nClass distribution:")
print(urldata['Type'].value_counts())

# --- SHUFFLE DATA ---
urldata = urldata.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nFinal dataset size:", len(urldata))
print(urldata.head())

Removed 14062 duplicate URLs

Class distribution:
Type
0    438292
1     52629
Name: count, dtype: int64

Final dataset size: 490921
                                                 url  Type
0  https://ticketcity.com/nfl-tickets/afc/cincinn...     0
1  https://lbdedetizadora.com.br/wp-admin/js/payp...     0
2                     https://webmailusal.weebly.com     1
3                 https://ptt-gov-tr.christmas/index     1
4  http://floreswindows.com/libraries/fof/hal/ren...     0


### 6. FEATURE EXTRACTION 

In [19]:
import re

# --- FEATURE FUNCTIONS ---

def get_length(url):
    return len(str(url))

def count_dots(url):
    return str(url).count('.')

def has_at_symbol(url):
    return 1 if '@' in str(url) else 0

def has_ip(url):
    pattern = r'(([01]?\d\d?|2[0-4]\d|25[0-5])\.){3}([01]?\d\d?|2[0-4]\d|25[0-5])'
    return 1 if re.search(pattern, str(url)) else 0

def count_hyphen(url):
    return str(url).count('-')

def count_slash(url):
    return str(url).count('/')

def check_https(url):
    return 0 if str(url).lower().startswith("https") else 1

def count_digits(url):
    return sum(c.isdigit() for c in str(url))

def is_shortened(url):
    match = re.search(
        r'bit\.ly|goo\.gl|tinyurl|t\.co|ow\.ly|is\.gd|buff\.ly|adf\.ly',
        str(url)
    )
    return 1 if match else 0

'''
from urllib.parse import urlparse

def get_domain(url):
    try:
        return urlparse(str(url)).netloc
    except:
        return ""   # fallback for invalid URLs

def domain_length(url):
    try:
        return len(get_domain(url))
    except:
        return 0
    
def subdomain_count(url):
    try:
        domain = get_domain(url)
        return domain.count('.')
    except:
        return 0

def has_suspicious_words(url):
    keywords = ['login', 'verify', 'bank', 'secure', 'account', 'update']
    return 1 if any(word in url.lower() for word in keywords) else 0

def suspicious_tld(url):
    tlds = ['.xyz', '.tk', '.ml', '.ga', '.cf']
    return 1 if any(tld in url.lower() for tld in tlds) else 0
    '''

# --- APPLY FEATURES TO urldata ---

urldata['url_length']   = urldata['url'].apply(get_length)
urldata['dot_count']    = urldata['url'].apply(count_dots)
urldata['has_at']       = urldata['url'].apply(has_at_symbol)
urldata['has_ip']       = urldata['url'].apply(has_ip)
urldata['hyphen_count'] = urldata['url'].apply(count_hyphen)
urldata['slash_count']  = urldata['url'].apply(count_slash)
urldata['has_https']    = urldata['url'].apply(check_https)
urldata['digit_count']  = urldata['url'].apply(count_digits)
urldata['is_shortened'] = urldata['url'].apply(is_shortened)
'''
urldata['domain_length'] = urldata['url'].apply(domain_length)
urldata['subdomain_count'] = urldata['url'].apply(subdomain_count)
urldata['has_suspicious_words'] = urldata['url'].apply(has_suspicious_words)
urldata['suspicious_tld'] = urldata['url'].apply(suspicious_tld)
'''

# --- VERIFY ---
print(urldata.head())
print("\nColumns:", urldata.columns)

                                                 url  Type  url_length  \
0  https://ticketcity.com/nfl-tickets/afc/cincinn...     0          70   
1  https://lbdedetizadora.com.br/wp-admin/js/payp...     0         130   
2                     https://webmailusal.weebly.com     1          30   
3                 https://ptt-gov-tr.christmas/index     1          34   
4  http://floreswindows.com/libraries/fof/hal/ren...     0          51   

   dot_count  has_at  has_ip  hyphen_count  slash_count  has_https  \
0          2       0       0             3            5          0   
1          5       0       0             2           11          0   
2          2       0       0             0            2          0   
3          1       0       0             2            3          0   
4          1       0       0             0            7          1   

   digit_count  is_shortened  
0            0             0  
1            6             0  
2            0             0  
3         

### 7. Train Test Split

In [20]:
from sklearn.model_selection import train_test_split

# --- FEATURES (X) and LABEL (y) ---
X = urldata.drop(['url', 'Type'], axis=1)
y = urldata['Type']

# --- TRAIN TEST SPLIT (IMPORTANT: stratify) ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,          # ⚠️ VERY IMPORTANT
    random_state=42
)

# --- VERIFY ---
print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

Training set size: (392736, 9)
Testing set size: (98185, 9)

Training class distribution:
Type
0    350633
1     42103
Name: count, dtype: int64

Testing class distribution:
Type
0    87659
1    10526
Name: count, dtype: int64


8. MODEL TRAINING

In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- MODEL ---
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    class_weight='balanced',   # 🔥 VERY IMPORTANT
    random_state=42,
    n_jobs=-1
)

# --- TRAIN ---
model.fit(X_train, y_train)

# --- PREDICT ---
y_pred = model.predict(X_test)

# --- EVALUATION ---
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9231043438407088

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.94      0.96     87659
           1       0.60      0.82      0.70     10526

    accuracy                           0.92     98185
   macro avg       0.79      0.88      0.83     98185
weighted avg       0.94      0.92      0.93     98185


Confusion Matrix:
[[82016  5643]
 [ 1907  8619]]


9. IDK.

In [22]:
def predict_url(url, model):
    url = url.lower().strip()
    url = url.replace('\n', '').replace('\r', '')
    url = url.replace("www.", "")
    if url.endswith('/'):
        url = url[:-1]

    features = {
        'url_length': get_length(url),
        'dot_count': count_dots(url),
        'has_at': has_at_symbol(url),
        'has_ip': has_ip(url),
        'hyphen_count': count_hyphen(url),
        'slash_count': count_slash(url),
        'has_https': check_https(url),
        'digit_count': count_digits(url),
        'is_shortened': is_shortened(url),
        #'domain_length': domain_length(url),
        #'subdomain_count': subdomain_count(url),
        #'has_suspicious_words': has_suspicious_words(url),
        #'suspicious_tld': suspicious_tld(url)
    }

    features_df = pd.DataFrame([features])

    prob = model.predict_proba(features_df)[0][1]

    # 🔥 stricter threshold
    if prob > 0.4:
        return f"Phishing 🚨 (confidence: {prob:.2f})"
    else:
        return f"Legitimate ✅ (confidence: {1-prob:.2f})"

In [23]:
print(predict_url("https://google.com", model))
print(predict_url("https://facebook.com", model))
print(predict_url("http://secure-login-alert-bank.xyz", model))
print(predict_url("http://bank-verification-secure-update.xyz", model))


Phishing 🚨 (confidence: 0.72)
Legitimate ✅ (confidence: 0.65)
Phishing 🚨 (confidence: 0.77)
Phishing 🚨 (confidence: 0.81)
